In [1]:
import pandas as pd

In [2]:
event_log = pd.read_parquet("~/jax_lob_sim/data/parquet/event_log/PFE_2504.parquet")
book_state = pd.read_parquet("~/jax_lob_sim/data/parquet/book_state/PFE_2504.parquet")
event_log.head()

,ts,level,size,action
0,1743516000018420653,-1,700,T
1,1743516000018471125,-1,100,A
2,1743516000018484476,-10,125,A
3,1743516000018572050,2,100,C
4,1743516000018682141,-1,100,A


In [3]:
book_state.head()

,ts,spread,imbalance,best_size,q-4,q-3,q-2,q-1,q+1,q+2,q+3,q+4,best_bid_px,best_ask_px
0,1743516000018317109,1,0.0704,1506,500,1210,1000,806,700,525,600,800,24.82,24.83
1,1743516000018420653,2,0.2111,1331,500,1210,1000,806,525,600,800,600,24.82,24.84
2,1743516000018471125,2,0.2662,1431,500,1210,1000,906,525,600,800,600,24.82,24.84
3,1743516000018484476,2,0.2662,1431,500,1210,1000,906,525,600,800,600,24.82,24.84
4,1743516000018572050,2,0.2662,1431,500,1210,1000,906,525,500,800,600,24.82,24.84


In [5]:
df = pd.merge(book_state, event_log, on="ts")

In [6]:
import numpy as np

def get_imbalance_bin(series):
    v = series.to_numpy()
    
    # Divide by 0.1 and round to fix floating-point precision issues
    # e.g., -0.1 / 0.1 could be -0.9999... instead of -1.0
    scaled = np.round(v / 0.1, 8)

    result = np.where(
        v == 0,                          # bin 0: exactly zero
        0,
        np.where(
            v < 0,
            np.floor(scaled).astype(int),  # negative: [0.1*i, 0.1*(i+1))
            np.ceil(scaled).astype(int)    # positive: (0.1*(i-1), 0.1*i]
        )
    )
    return result

df['imbalance_bin'] = get_imbalance_bin(df['imbalance'])

In [5]:
book_state.head()

,ts,spread,imbalance,best_size,q-4,q-3,q-2,q-1,q+1,q+2,q+3,q+4,best_bid_px,best_ask_px,imbalance_bin
0,1743516000018317109,1,0.0704,1506,500,1210,1000,806,700,525,600,800,24.82,24.83,1
1,1743516000018420653,2,0.2111,1331,500,1210,1000,806,525,600,800,600,24.82,24.84,3
2,1743516000018471125,2,0.2662,1431,500,1210,1000,906,525,600,800,600,24.82,24.84,3
3,1743516000018484476,2,0.2662,1431,500,1210,1000,906,525,600,800,600,24.82,24.84,3
4,1743516000018572050,2,0.2662,1431,500,1210,1000,906,525,500,800,600,24.82,24.84,3


In [13]:
book_state["imbalance_bin"].value_counts()

imbalance_bin
-4     6914
-3     6757
-2     6402
 1     6258
-5     6250
-1     6216
 2     6037
 3     5659
-6     5513
 4     4923
-7     4176
-8     4145
 5     3999
 6     3584
-9     3479
-10    3301
 7     2784
 8     1925
 9     1686
 10    1386
 0      118
Name: count, dtype: int64

In [ ]:
def estimate_event_probs(df: pd.Series) -> np.array:
    

In [13]:
df[df["action"]=="A"]["level"].value_counts()

level
 1     20246
-1     14430
 2      2338
-2      2239
 4      1350
-4      1278
 3      1262
-3      1206
 5       898
-5       785
 7       158
-7       143
 6       142
 8       140
-8       118
-6       109
 10       65
-9        54
-10       52
 9        32
Name: count, dtype: int64

In [15]:
len(df[(df["level"]<=2)&(df["level"]>=-2)].groupby(["level", "action"]))

12

In [7]:
def estimate_intensities(df):
    # 1. Group by both columns and calculate the mean of the time differences
    # We use a lambda to handle the diff().mean() calculation per group
    grouped = df.groupby(["imbalance_bin", "spread"])["ts"].apply(
        lambda x: x.diff().mean()
    )

    # 2. Unstack 'spread' to the columns to get the shape (imbalance_bins, spreads)
    # .fillna(0) handles any missing combinations where no events occurred
    matrix_df = grouped.unstack(level="spread").fillna(0)

    # 3. Convert directly to a NumPy array
    return matrix_df.to_numpy()
        

In [8]:
vector = estimate_intensities(df)
vector.shape

(21, 3)

In [ ]:
for bin, temp_df in book_state.groupby(["spread", "imbalance_bin"])["ts"]:
    temp_ts = ts

type(temp_ts)

pandas.core.series.Series

In [22]:
temp_ts.to_list()

[1743516002520153646, 1743516002520201167, 1743516002520251639]

In [26]:
event_log[event_log["ts"].isin(temp_ts)]

,ts,level,size,action
299,1743516002520153646,-1,200,A
300,1743516002520201167,-3,60,C
301,1743516002520251639,-1,100,A


In [28]:
temp_ts.diff().dropna().mean()

np.float64(48996.5)

In [29]:
event_log["action"].value_counts()

action
A             47045
C             44499
T              3371
CREATE_BID       25
CREATE_ASK       20
Name: count, dtype: int64

In [30]:
event_log[event_log["action"]=="T"]

,ts,level,size,action
0,1743516000018420653,-1,700,T
47,1743516000135742305,1,212,T
48,1743516000135742305,1,200,T
49,1743516000135742305,1,44,T
58,1743516000136792289,1,100,T
...,...,...,...,...
94789,1743526990989521633,-1,89,T
94790,1743526990989521633,-1,100,T
94791,1743526990989521633,-1,200,T
94826,1743526991080081029,1,1,T


In [34]:
event_log[event_log["ts"]==1743516000018420653]

,ts,level,size,action
0,1743516000018420653,-1,700,T


In [35]:
book_state.iloc[45:50]

,ts,spread,imbalance,best_size,q-4,q-3,q-2,q-1,q+1,q+2,q+3,q+4,best_bid_px,best_ask_px,imbalance_bin
45,1743516000134792467,1,-0.6385,2523,1210,1000,714,456,2067,600,800,600,24.83,24.84,-7
46,1743516000134804141,1,-0.6818,2866,1210,1000,714,456,2410,600,800,600,24.83,24.84,-7
47,1743516000135720222,1,-0.6385,2523,1210,1000,714,456,2067,600,800,600,24.83,24.84,-7
48,1743516000135742305,2,-0.4865,2781,500,1210,1000,714,2067,600,800,600,24.82,24.84,-5
49,1743516000135779053,2,-0.5043,2881,500,1210,1000,714,2167,600,800,600,24.82,24.84,-6


In [ ]:
"""
Event types (12 total)
ADD_BID_L2, ADD_BID_L1, ADD_ASK_L1, ADD_ASK_L2 : 0~3
CANCEL_BID_L2, CANCEL_BID_L1, CANCEL_ASK_L1, CANCEL_ASK_L2 : 5~8
TRADE_BID, TRADE_ASK : 9, 10
CREATE_BID, CREATE_ASK : 11, 12
"""
mapping_level = {-2: 0, -1: 1, 1: 2, 2: 3}
mapping_event = {"A": 2, "C": 7, }